# Soal 2 · Monitor Cluster dan Task (sisi Master)
**Big Data** · dijalankan di **JupyterLab PC Master** (192.168.1.15), di tab terpisah dari `Soal2_RDD.ipynb`.

Diadaptasi dari materi **RDD-Master.ipynb**. Notebook ini **tidak menjalankan Spark**; ia hanya:
1. menampilkan status cluster dari Spark Master UI (`:8080/json/`): Worker ALIVE, core, memori, aplikasi aktif;
2. menerima laporan task **RUNNING/DONE** dari executor (server port **8090**) saat Sel 9 `Soal2_RDD.ipynb` dijalankan.

Peran **Worker** dipegang layanan **Spark Worker** di keempat PC; kode yang mereka jalankan dikirim oleh driver (`Soal2_RDD.ipynb`).

Urutan: **Sel 1** sekali → **Sel 2** (laporan cluster untuk laporan) → **Sel 3** (monitor, hentikan dengan ■). Server tetap hidup setelah monitor dihentikan.

## Sel 1. Server laporan task (jalankan sekali)

In [1]:
# ===== SEL 1: Server laporan task (port 8090) =====
import json, time, threading
from collections import defaultdict
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.request import urlopen

MASTER  = "192.168.1.15"     # IP PC Master
PORT    = 8090
REFRESH = 2

try:
    laporan
except NameError:
    laporan = {}
    lock = threading.Lock()

class TaskHandler(BaseHTTPRequestHandler):
    def do_POST(self):
        try:
            size = int(self.headers.get("Content-Length", 0))
            data = json.loads(self.rfile.read(size))
            if not all(k in data for k in ("run", "task", "worker", "app", "status")):
                self.send_error(400, "Field laporan tidak lengkap"); return
            data["waktu"] = time.time()
            with lock:
                laporan[(data["run"], data["task"], data["worker"])] = data
            self.send_response(200); self.send_header("Content-Length", "2"); self.end_headers()
            self.wfile.write(b"OK")
        except Exception:
            self.send_error(400, "Format laporan tidak valid")

    def do_GET(self):
        with lock:
            n = len(laporan)
        body = f"Monitor aktif. Jumlah laporan: {n}\n".encode()
        self.send_response(200); self.send_header("Content-Length", str(len(body))); self.end_headers()
        self.wfile.write(body)

    def log_message(self, *args):
        pass

try:
    server.shutdown(); server.server_close()
except NameError:
    pass
ThreadingHTTPServer.allow_reuse_address = True
server = ThreadingHTTPServer(("0.0.0.0", PORT), TaskHandler)
threading.Thread(target=server.serve_forever, daemon=True).start()
print(f"Server laporan berjalan di http://{MASTER}:{PORT}")
print("Uji dari PC Worker (PowerShell): Test-NetConnection", MASTER, "-Port", PORT)

Server laporan berjalan di http://192.168.1.15:8090
Uji dari PC Worker (PowerShell): Test-NetConnection 192.168.1.15 -Port 8090


## Sel 2. Laporan cluster (bukti 4 Worker untuk laporan)

In [2]:
# ===== SEL 2: Laporan cluster dari Spark Master UI =====
def get_json(url):
    try:
        with urlopen(url, timeout=3) as r:
            return json.load(r)
    except Exception:
        return None

def laporan_cluster():
    d = get_json(f"http://{MASTER}:8080/json/")
    if not d:
        print("Spark Master UI tidak dapat diakses (cek Spark Master & port 8080)."); return
    w = d.get("workers", [])
    hidup = [x for x in w if x.get("state") == "ALIVE"]
    print(f"Master {d.get('url', '')} | status {d.get('status')} | Worker ALIVE {len(hidup)}/{len(w)} | "
          f"core {d.get('coresused', 0)}/{d.get('cores', 0)} dipakai | memori {d.get('memoryused', 0)}/{d.get('memory', 0)} MB")
    print(f"\n{'WORKER (host)':<18}{'STATUS':<8}{'CORE':>6}{'DIPAKAI':>9}{'MEMORI MB':>11}{'DIPAKAI':>9}")
    for x in sorted(w, key=lambda x: x.get("host", "")):
        print(f"{x.get('host', '?'):<18}{x.get('state', '?'):<8}{x.get('cores', 0):>6}{x.get('coresused', 0):>9}"
              f"{x.get('memory', 0):>11}{x.get('memoryused', 0):>9}")
    apps = d.get("activeapps", [])
    print(f"\nAplikasi aktif: {len(apps)}")
    for a in apps:
        print(f"  {a.get('name')} | {a.get('id')} | core {a.get('cores')} | state {a.get('state')}")
    alamat = {x.get("host") for x in hidup}
    print("\nWorker berada di", len(alamat), "alamat IP berbeda:", sorted(alamat))

laporan_cluster()

Master spark://192.168.1.15:7077 | status ALIVE | Worker ALIVE 4/4 | core 0/16 dipakai | memori 0/21504 MB

WORKER (host)     STATUS    CORE  DIPAKAI  MEMORI MB  DIPAKAI
192.168.1.13      ALIVE        4        0       3072        0
192.168.1.14      ALIVE        4        0       6144        0
192.168.1.19      ALIVE        4        0       6144        0
192.168.1.9       ALIVE        4        0       6144        0

Aplikasi aktif: 0

Worker berada di 4 alamat IP berbeda: ['192.168.1.13', '192.168.1.14', '192.168.1.19', '192.168.1.9']


## Sel 3. Monitor task per operasi dan per Worker (hentikan dengan ■)

In [4]:
# ===== SEL 3: Monitor task =====
from IPython.display import clear_output

def tampilkan():
    print("=" * 85)             
    print(" MONITOR OPERASI RDD - SPARK CLUSTER (Soal 2)")
    print("=" * 85)
    laporan_cluster()
    with lock:
        snap = list(laporan.values())
    print("\n--- RINGKASAN PER OPERASI (RUNNING / DONE) ---")
    if not snap:
        print("Belum ada laporan task (jalankan Sel 9 di Soal2_RDD.ipynb).")
        return
    per_op, per_worker = defaultdict(lambda: [0, 0]), defaultdict(lambda: [0, 0])
    for t in snap:
        s = str(t.get("status", "")).upper()
        i = 0 if s == "RUNNING" else 1 if s == "DONE" else None
        if i is None:
            continue
        per_op[t["app"]][i] += 1
        per_worker[t["worker"]][i] += 1
    for op, (r, d) in sorted(per_op.items()):
        print(f"{op:<45} RUNNING {r:<4} DONE {d:<4}")
    print("\n--- PER WORKER (RUNNING / DONE) ---")
    for w, (r, d) in sorted(per_worker.items()):
        print(f"{w:<28} {r:>3} / {d:<3}")

try:
    while True:
        clear_output(wait=True)
        tampilkan()
        print(f"\nRefresh {REFRESH} detik | Interrupt (■) untuk berhenti")
        time.sleep(REFRESH)
except KeyboardInterrupt:
    print("\nMonitor dihentikan (server laporan tetap berjalan).")

 MONITOR OPERASI RDD - SPARK CLUSTER (Soal 2)
Master spark://192.168.1.15:7077 | status ALIVE | Worker ALIVE 4/4 | core 16/16 dipakai | memori 8192/21504 MB

WORKER (host)     STATUS    CORE  DIPAKAI  MEMORI MB  DIPAKAI
192.168.1.13      ALIVE        4        4       3072     2048
192.168.1.14      ALIVE        4        4       6144     2048
192.168.1.19      ALIVE        4        4       6144     2048
192.168.1.9       ALIVE        4        4       6144     2048

Aplikasi aktif: 1
  Soal2d-lineage-cache | app-20261118211006-0040 | core 16 | state RUNNING

Worker berada di 4 alamat IP berbeda: ['192.168.1.13', '192.168.1.14', '192.168.1.19', '192.168.1.9']

--- RINGKASAN PER OPERASI (RUNNING / DONE) ---
Belum ada laporan task (jalankan Sel 9 di Soal2_RDD.ipynb).

Refresh 2 detik | Interrupt (■) untuk berhenti

Monitor dihentikan (server laporan tetap berjalan).


## Sel 4 (opsional). Kosongkan laporan / matikan server

In [3]:
# ===== SEL 4: Reset laporan =====
with lock:
    laporan.clear()
print("Laporan dikosongkan.")
# server.shutdown(); server.server_close()   # buka komentar untuk mematikan server

Laporan dikosongkan.
